# Dance Library - Storage & Database Integrity Check

Checks the three-way relationship between **Supabase DB records**, **R2 videos**, and **R2 thumbnails** by cross-referencing UUIDs.

Every entity in the system is identified by a UUID. For each UUID, three things can exist:

| DB Record | Video in R2 | Thumbnail in R2 | Status |
|-----------|-------------|-----------------|--------|
| Y | Y | Y | **Perfect** — fully synced |
| Y | Y | N | **Missing thumbnail** — video works, but no thumb |
| Y | N | Y | **Missing video** — broken record, thumb is orphaned |
| Y | N | N | **Ghost record** — DB row with nothing in storage |
| N | Y | Y | **Orphaned pair** — R2 files with no DB record |
| N | Y | N | **Orphaned video** — lone video, no DB or thumb |
| N | N | Y | **Orphaned thumbnail** — lone thumb, no DB or video |

Run all cells to get a full diagnostic. Cleanup actions at the bottom are **commented out** — review before uncommenting.

In [1]:
import os
from dotenv import load_dotenv

# Load from .env file if it exists, otherwise set manually below
load_dotenv()

def clean_env(key):
    """Get env var and strip any inline # comments that dotenv may include."""
    val = os.getenv(key, "")
    return val.split("#")[0].strip() if val else ""

# ── Supabase config ──
SUPABASE_URL = clean_env("VITE_SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = clean_env("VITE_SUPABASE_SERVICE_ROLE_KEY")

# ── R2 config ──
R2_ACCOUNT_ID = clean_env("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = clean_env("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = clean_env("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME = clean_env("R2_BUCKET_NAME")

# Validate
missing = [k for k, v in {
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "R2_ACCOUNT_ID": R2_ACCOUNT_ID,
    "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
    "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    "R2_BUCKET_NAME": R2_BUCKET_NAME,
}.items() if not v]

if missing:
    print(f"Missing config: {', '.join(missing)}")
    print("Set them above or create a .env file in the TESTING folder.")
else:
    print("All config loaded.")

All config loaded.


In [2]:
import boto3
from supabase import create_client

# ── Connect to Supabase ──
sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print("Supabase connected.")

# ── Connect to R2 via S3 API ──
r2 = boto3.client(
    service_name="s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)
print("R2 connected.")

Supabase connected.
R2 connected.


In [3]:
# ── Fetch all DB media records ──
import os

db_records = []
page_size = 1000
offset = 0

while True:
    res = sb.table("media").select(
        "id, title, storage_path, thumbnail_path, original_filename, file_size_bytes, mime_type, duration, created_at"
    ).range(offset, offset + page_size - 1).execute()
    db_records.extend(res.data)
    if len(res.data) < page_size:
        break
    offset += page_size

print(f"Found {len(db_records)} media records in database.")

# Build UUID -> record lookup
# storage_path format: "videos/{uuid}.MOV"
def uuid_from_path(path: str) -> str:
    return os.path.splitext(path.split("/")[-1])[0]

db_by_uuid = {}
for r in db_records:
    uid = uuid_from_path(r["storage_path"])
    db_by_uuid[uid] = r

Found 481 media records in database.


In [4]:
# ── List all R2 objects and build UUID lookups ──
r2_objects = {}  # key -> size_bytes
continuation_token = None

while True:
    kwargs = {"Bucket": R2_BUCKET_NAME, "MaxKeys": 1000}
    if continuation_token:
        kwargs["ContinuationToken"] = continuation_token

    response = r2.list_objects_v2(**kwargs)

    for obj in response.get("Contents", []):
        r2_objects[obj["Key"]] = obj["Size"]

    if response.get("IsTruncated"):
        continuation_token = response["NextContinuationToken"]
    else:
        break

r2_keys = set(r2_objects.keys())
r2_video_keys = {k for k in r2_keys if k.startswith("videos/")}
r2_thumb_keys = {k for k in r2_keys if k.startswith("thumbs/")}

print(f"Found {len(r2_keys)} total objects in R2:")
print(f"  - {len(r2_video_keys)} videos")
print(f"  - {len(r2_thumb_keys)} thumbnails")
print(f"  - {len(r2_keys - r2_video_keys - r2_thumb_keys)} other")

# Build UUID -> R2 key lookups
r2_videos_by_uuid = {}  # uuid -> key
r2_thumbs_by_uuid = {}  # uuid -> key

for k in r2_video_keys:
    uid = uuid_from_path(k)
    r2_videos_by_uuid[uid] = k

for k in r2_thumb_keys:
    uid = uuid_from_path(k)
    r2_thumbs_by_uuid[uid] = k

# Collect all known UUIDs across all three sources
all_uuids = set(db_by_uuid.keys()) | set(r2_videos_by_uuid.keys()) | set(r2_thumbs_by_uuid.keys())
print(f"\nUnique UUIDs across all sources: {len(all_uuids)}")

Found 966 total objects in R2:
  - 484 videos
  - 482 thumbnails
  - 0 other

Unique UUIDs across all sources: 484


In [5]:
from tabulate import tabulate

def format_bytes(size):
    for unit in ["B", "KB", "MB", "GB"]:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"

# ══════════════════════════════════════════════
# Classify every UUID into one of 7 states
# ══════════════════════════════════════════════
states = {
    "perfect": [],          # DB + Video + Thumb
    "missing_thumb": [],    # DB + Video, no Thumb
    "missing_video": [],    # DB + Thumb, no Video
    "ghost_record": [],     # DB only, nothing in R2
    "orphaned_pair": [],    # Video + Thumb, no DB
    "orphaned_video": [],   # Video only, no DB or Thumb
    "orphaned_thumb": [],   # Thumb only, no DB or Video
}

for uid in sorted(all_uuids):
    has_db = uid in db_by_uuid
    has_video = uid in r2_videos_by_uuid
    has_thumb = uid in r2_thumbs_by_uuid

    if has_db and has_video and has_thumb:
        states["perfect"].append(uid)
    elif has_db and has_video and not has_thumb:
        states["missing_thumb"].append(uid)
    elif has_db and not has_video and has_thumb:
        states["missing_video"].append(uid)
    elif has_db and not has_video and not has_thumb:
        states["ghost_record"].append(uid)
    elif not has_db and has_video and has_thumb:
        states["orphaned_pair"].append(uid)
    elif not has_db and has_video and not has_thumb:
        states["orphaned_video"].append(uid)
    elif not has_db and not has_video and has_thumb:
        states["orphaned_thumb"].append(uid)

# ── Overview ──
print("=" * 60)
print("INTEGRITY ANALYSIS (by UUID)")
print("=" * 60)
print()

status_rows = [
    ["Perfect (DB + Video + Thumb)", len(states["perfect"])],
    ["Missing thumbnail (DB + Video)", len(states["missing_thumb"])],
    ["Missing video (DB + Thumb)", len(states["missing_video"])],
    ["Ghost record (DB only)", len(states["ghost_record"])],
    ["Orphaned pair (Video + Thumb)", len(states["orphaned_pair"])],
    ["Orphaned video only", len(states["orphaned_video"])],
    ["Orphaned thumbnail only", len(states["orphaned_thumb"])],
]
print(tabulate(status_rows, headers=["State", "Count"], tablefmt="simple"))

total_issues = sum(len(v) for k, v in states.items() if k != "perfect")
print(f"\nTotal UUIDs: {len(all_uuids)}  |  Issues: {total_issues}")

INTEGRITY ANALYSIS (by UUID)

State                             Count
------------------------------  -------
Perfect (DB + Video + Thumb)        481
Missing thumbnail (DB + Video)        0
Missing video (DB + Thumb)            0
Ghost record (DB only)                0
Orphaned pair (Video + Thumb)         1
Orphaned video only                   2
Orphaned thumbnail only               0

Total UUIDs: 484  |  Issues: 3


In [6]:
# ══════════════════════════════════════════════
# Detailed breakdown of issues (skips "perfect")
# ══════════════════════════════════════════════

def print_issue_section(label, uuids, show_db=True, show_r2=True):
    if not uuids:
        return
    print(f"\n{'─' * 60}")
    print(f"{label} ({len(uuids)})")
    print(f"{'─' * 60}")
    rows = []
    for uid in uuids:
        row = [uid[:12] + "..."]
        if show_db and uid in db_by_uuid:
            r = db_by_uuid[uid]
            row.append(r.get("title", "N/A")[:35])
            row.append(r.get("original_filename", "N/A")[:30])
        elif show_db:
            row.extend(["—", "—"])
        if show_r2:
            vid_size = format_bytes(r2_objects[r2_videos_by_uuid[uid]]) if uid in r2_videos_by_uuid else "—"
            thm_size = format_bytes(r2_objects[r2_thumbs_by_uuid[uid]]) if uid in r2_thumbs_by_uuid else "—"
            row.extend([vid_size, thm_size])
        rows.append(row)

    headers = ["UUID"]
    if show_db:
        headers += ["Title", "Filename"]
    if show_r2:
        headers += ["Video Size", "Thumb Size"]
    print(tabulate(rows, headers=headers, tablefmt="simple"))

print_issue_section("MISSING THUMBNAIL — DB + Video, no Thumb", states["missing_thumb"])
print_issue_section("MISSING VIDEO — DB + Thumb, no Video (broken)", states["missing_video"])
print_issue_section("GHOST RECORD — DB only, nothing in R2", states["ghost_record"], show_r2=False)
print_issue_section("ORPHANED PAIR — Video + Thumb, no DB record", states["orphaned_pair"], show_db=False)
print_issue_section("ORPHANED VIDEO — Video only, no DB or Thumb", states["orphaned_video"], show_db=False)
print_issue_section("ORPHANED THUMBNAIL — Thumb only, no DB or Video", states["orphaned_thumb"], show_db=False)

if total_issues == 0:
    print("\nNo issues found. Everything is in sync!")


────────────────────────────────────────────────────────────
ORPHANED PAIR — Video + Thumb, no DB record (1)
────────────────────────────────────────────────────────────
UUID             Video Size    Thumb Size
---------------  ------------  ------------
2c48cf49-625...  140.4 MB      15.3 KB

────────────────────────────────────────────────────────────
ORPHANED VIDEO — Video only, no DB or Thumb (2)
────────────────────────────────────────────────────────────
UUID             Video Size    Thumb Size
---------------  ------------  ------------
0a19dbdd-955...  140.4 MB      —
5372fbe3-432...  3.2 MB        —


In [7]:
# ══════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════
total_r2_size = sum(r2_objects.values())

# Calculate wasted storage (orphaned files)
orphaned_uuids = states["orphaned_pair"] + states["orphaned_video"] + states["orphaned_thumb"]
orphaned_storage = 0
for uid in orphaned_uuids:
    if uid in r2_videos_by_uuid:
        orphaned_storage += r2_objects[r2_videos_by_uuid[uid]]
    if uid in r2_thumbs_by_uuid:
        orphaned_storage += r2_objects[r2_thumbs_by_uuid[uid]]

print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"")
print(f"Database records:       {len(db_records)}")
print(f"R2 objects:             {len(r2_keys)}")
print(f"Total R2 storage:       {format_bytes(total_r2_size)}")
print(f"Orphaned storage:       {format_bytes(orphaned_storage)}")
print(f"")
print(f"Perfect:                {len(states['perfect'])}")
print(f"Issues:                 {total_issues}")
if total_issues:
    print(f"  Missing thumbnails:   {len(states['missing_thumb'])}")
    print(f"  Missing videos:       {len(states['missing_video'])}")
    print(f"  Ghost records:        {len(states['ghost_record'])}")
    print(f"  Orphaned pairs:       {len(states['orphaned_pair'])}")
    print(f"  Orphaned videos:      {len(states['orphaned_video'])}")
    print(f"  Orphaned thumbnails:  {len(states['orphaned_thumb'])}")
print(f"")
if total_issues == 0:
    print("Everything is in sync!")
else:
    print("Issues found — review the detailed breakdown above.")

SUMMARY

Database records:       481
R2 objects:             966
Total R2 storage:       55.0 GB
Orphaned storage:       284.0 MB

Perfect:                481
Issues:                 3
  Missing thumbnails:   0
  Missing videos:       0
  Ghost records:        0
  Orphaned pairs:       1
  Orphaned videos:      2
  Orphaned thumbnails:  0

Issues found — review the detailed breakdown above.


---
## File Type Audit

Cross-references file extensions and MIME types across R2 objects and Supabase records to find mismatches:

| Check | What it finds |
|-------|---------------|
| R2 extensions | All unique file extensions stored in R2 (`videos/` and `thumbs/`) |
| DB `mime_type` values | All unique MIME types recorded in the database |
| DB `media_type` values | All unique media type classifications (`video`, `image`, etc.) |
| Extension vs MIME mismatch | Files where the R2 extension doesn't match the DB MIME type |
| Extension vs media_type mismatch | Files classified as `video` but with image extensions (or vice versa) |
| R2 Content-Type | Actual Content-Type headers stored on R2 objects (sampled) |

In [8]:
from collections import defaultdict

# ══════════════════════════════════════════════
# FILE TYPE AUDIT
# ══════════════════════════════════════════════

# Known mappings: extension -> expected MIME type prefix
EXT_TO_MIME = {
    ".mov": "video/",  ".mp4": "video/",  ".m4v": "video/",
    ".avi": "video/",  ".mkv": "video/",  ".webm": "video/",
    ".3gp": "video/",  ".hevc": "video/",
    ".heic": "image/", ".jpg": "image/",  ".jpeg": "image/",
    ".png": "image/",  ".gif": "image/",  ".webp": "image/",
    ".mp3": "audio/",  ".wav": "audio/",  ".m4a": "audio/",
}

EXT_TO_MEDIA_TYPE = {
    ".mov": "video",  ".mp4": "video",  ".m4v": "video",
    ".avi": "video",  ".mkv": "video",  ".webm": "video",
    ".3gp": "video",  ".hevc": "video",
    ".heic": "image", ".jpg": "image",  ".jpeg": "image",
    ".png": "image",  ".gif": "image",  ".webp": "image",
    ".mp3": "audio",  ".wav": "audio",  ".m4a": "audio",
}

# ── 1. Unique extensions in R2 ──
r2_video_exts = defaultdict(int)
for key in r2_video_keys:
    ext = os.path.splitext(key)[1].lower()
    r2_video_exts[ext] += 1

r2_thumb_exts = defaultdict(int)
for key in r2_thumb_keys:
    ext = os.path.splitext(key)[1].lower()
    r2_thumb_exts[ext] += 1

print("=" * 60)
print("FILE TYPE AUDIT")
print("=" * 60)

print(f"\n{'─' * 60}")
print("R2 file extensions")
print(f"{'─' * 60}")
print("\n  videos/ folder:")
for ext, count in sorted(r2_video_exts.items(), key=lambda x: -x[1]):
    print(f"    {ext:10s} {count:>5d}")
print(f"\n  thumbs/ folder:")
for ext, count in sorted(r2_thumb_exts.items(), key=lambda x: -x[1]):
    print(f"    {ext:10s} {count:>5d}")

# ── 2. Unique DB mime_type and media_type values ──
db_mime_types = defaultdict(int)
for r in db_records:
    mime = r.get("mime_type") or "(null)"
    db_mime_types[mime] += 1

# We didn't fetch media_type in the initial query — fetch it now
media_type_res = sb.table("media").select("id, media_type").execute()
media_type_by_id = {r["id"]: r["media_type"] for r in media_type_res.data}

db_media_types = defaultdict(int)
for r in media_type_res.data:
    db_media_types[r["media_type"]] += 1

print(f"\n{'─' * 60}")
print("DB mime_type values")
print(f"{'─' * 60}")
for mime, count in sorted(db_mime_types.items(), key=lambda x: -x[1]):
    print(f"    {mime:35s} {count:>5d}")

print(f"\n{'─' * 60}")
print("DB media_type values")
print(f"{'─' * 60}")
for mt, count in sorted(db_media_types.items(), key=lambda x: -x[1]):
    print(f"    {mt:35s} {count:>5d}")

# ── 3. Mismatches: R2 extension vs DB mime_type ──
mime_mismatches = []
for uid in db_by_uuid:
    r = db_by_uuid[uid]
    if uid not in r2_videos_by_uuid:
        continue
    r2_key = r2_videos_by_uuid[uid]
    ext = os.path.splitext(r2_key)[1].lower()
    db_mime = r.get("mime_type") or ""
    expected_prefix = EXT_TO_MIME.get(ext, "")

    if expected_prefix and db_mime and not db_mime.startswith(expected_prefix):
        mime_mismatches.append({
            "uuid": uid,
            "title": r.get("title", "—"),
            "filename": r.get("original_filename", "—"),
            "r2_ext": ext,
            "db_mime": db_mime,
            "expected": expected_prefix + "*",
        })

# ── 4. Mismatches: R2 extension vs DB media_type ──
type_mismatches = []
for uid in db_by_uuid:
    r = db_by_uuid[uid]
    if uid not in r2_videos_by_uuid:
        continue
    r2_key = r2_videos_by_uuid[uid]
    ext = os.path.splitext(r2_key)[1].lower()
    db_media_type = media_type_by_id.get(r["id"], "")
    expected_type = EXT_TO_MEDIA_TYPE.get(ext, "")

    if expected_type and db_media_type and expected_type != db_media_type:
        type_mismatches.append({
            "uuid": uid,
            "title": r.get("title", "—"),
            "filename": r.get("original_filename", "—"),
            "r2_ext": ext,
            "db_media_type": db_media_type,
            "expected_type": expected_type,
        })

# ── 5. Sample R2 Content-Type headers for problematic files ──
# Check all files with mismatches + all non-standard extensions
check_uuids = set()
for m in mime_mismatches:
    check_uuids.add(m["uuid"])
for m in type_mismatches:
    check_uuids.add(m["uuid"])
# Also check files with unusual extensions
for uid, key in r2_videos_by_uuid.items():
    ext = os.path.splitext(key)[1].lower()
    if ext not in {".mov", ".mp4", ".webp"}:
        check_uuids.add(uid)

r2_content_types = {}
for uid in check_uuids:
    if uid in r2_videos_by_uuid:
        try:
            head = r2.head_object(Bucket=R2_BUCKET_NAME, Key=r2_videos_by_uuid[uid])
            r2_content_types[uid] = head.get("ContentType", "(unknown)")
        except Exception as e:
            r2_content_types[uid] = f"(error: {e})"

# ── Print mismatches ──
print(f"\n{'─' * 60}")
print(f"EXTENSION vs MIME TYPE MISMATCHES ({len(mime_mismatches)})")
print(f"{'─' * 60}")
if mime_mismatches:
    rows = []
    for m in mime_mismatches:
        r2_ct = r2_content_types.get(m["uuid"], "—")
        rows.append([
            m["filename"][:30], m["r2_ext"], m["db_mime"],
            m["expected"], r2_ct
        ])
    print(tabulate(rows, headers=["Filename", "R2 Ext", "DB MIME", "Expected", "R2 Content-Type"], tablefmt="simple"))
else:
    print("  None found.")

print(f"\n{'─' * 60}")
print(f"EXTENSION vs MEDIA_TYPE MISMATCHES ({len(type_mismatches)})")
print(f"{'─' * 60}")
if type_mismatches:
    rows = []
    for m in type_mismatches:
        r2_ct = r2_content_types.get(m["uuid"], "—")
        rows.append([
            m["filename"][:30], m["r2_ext"], m["db_media_type"],
            m["expected_type"], r2_ct
        ])
    print(tabulate(rows, headers=["Filename", "R2 Ext", "DB Type", "Expected", "R2 Content-Type"], tablefmt="simple"))
else:
    print("  None found.")

# ── Summary of files with non-standard R2 Content-Type ──
if r2_content_types:
    print(f"\n{'─' * 60}")
    print(f"R2 CONTENT-TYPE HEADERS (sampled: {len(r2_content_types)} files)")
    print(f"{'─' * 60}")
    rows = []
    for uid, ct in sorted(r2_content_types.items()):
        r = db_by_uuid.get(uid, {})
        rows.append([
            r.get("original_filename", "—")[:30],
            os.path.splitext(r2_videos_by_uuid.get(uid, ""))[1],
            r.get("mime_type", "—"),
            ct
        ])
    print(tabulate(rows, headers=["Filename", "R2 Ext", "DB MIME", "R2 Content-Type"], tablefmt="simple"))

total_mismatches = len(mime_mismatches) + len(type_mismatches)
if total_mismatches == 0:
    print("\nAll file types are consistent across R2 and database.")

FILE TYPE AUDIT

────────────────────────────────────────────────────────────
R2 file extensions
────────────────────────────────────────────────────────────

  videos/ folder:
    .mp4         281
    .mov         202
    .mkv           1

  thumbs/ folder:
    .webp        481
    .jpg           1

────────────────────────────────────────────────────────────
DB mime_type values
────────────────────────────────────────────────────────────
    video/mp4                             279
    video/quicktime                       202

────────────────────────────────────────────────────────────
DB media_type values
────────────────────────────────────────────────────────────
    video                                 481

────────────────────────────────────────────────────────────
EXTENSION vs MIME TYPE MISMATCHES (0)
────────────────────────────────────────────────────────────
  None found.

────────────────────────────────────────────────────────────
EXTENSION vs MEDIA_TYPE MISMATCHES (0

---
## Exact Duplicates

These are definite duplicates — multiple DB records that share the same identity. Safe to clean up (keep oldest, remove the rest).

| Check | What it finds | Severity |
|-------|---------------|----------|
| Duplicate filenames | Multiple DB records with the same `original_filename` | **High** — same file uploaded more than once |
| Duplicate storage paths | Two DB records pointing at the same R2 key | **Critical** — data corruption |

In [9]:
from collections import defaultdict

# ══════════════════════════════════════════════
# EXACT DUPLICATES
# ══════════════════════════════════════════════

# ── 1. Duplicate filenames in DB ──
fname_groups = defaultdict(list)
for r in db_records:
    fn = r.get("original_filename")
    if fn:
        fname_groups[fn].append(r)

dupe_filenames = {fn: recs for fn, recs in fname_groups.items() if len(recs) > 1}

# ── 2. Duplicate storage_path references ──
spath_groups = defaultdict(list)
for r in db_records:
    spath_groups[r["storage_path"]].append(r)

dupe_storage_paths = {sp: recs for sp, recs in spath_groups.items() if len(recs) > 1}

# ── Summary ──
print("=" * 60)
print("EXACT DUPLICATES")
print("=" * 60)
print()

exact_rows = [
    ["Duplicate filenames in DB", len(dupe_filenames)],
    ["Duplicate storage_path refs", len(dupe_storage_paths)],
]
print(tabulate(exact_rows, headers=["Check", "Groups Found"], tablefmt="simple"))

total_exact_dupes = len(dupe_filenames) + len(dupe_storage_paths)
extra_records = sum(len(recs) - 1 for recs in dupe_filenames.values()) + \
                sum(len(recs) - 1 for recs in dupe_storage_paths.values())
print(f"\nTotal exact duplicate groups: {total_exact_dupes}  ({extra_records} extra records)")

# ── Detailed output ──

if dupe_filenames:
    print(f"\n{'─' * 60}")
    print(f"DUPLICATE FILENAMES ({len(dupe_filenames)} groups)")
    print(f"{'─' * 60}")
    for fn, recs in sorted(dupe_filenames.items()):
        print(f"\n  Filename: {fn}  ({len(recs)} records)")
        rows = []
        for i, r in enumerate(sorted(recs, key=lambda x: x["created_at"])):
            uid = uuid_from_path(r["storage_path"])
            size = format_bytes(r["file_size_bytes"]) if r.get("file_size_bytes") else "—"
            marker = "KEEP" if i == 0 else "DUPE"
            rows.append([marker, uid[:12] + "...", r.get("title", "—")[:35], size, r["created_at"][:19]])
        print(tabulate(rows, headers=["", "UUID", "Title", "File Size", "Created At"], tablefmt="simple"))

if dupe_storage_paths:
    print(f"\n{'─' * 60}")
    print(f"DUPLICATE STORAGE PATHS ({len(dupe_storage_paths)} groups) — DATA CORRUPTION!")
    print(f"{'─' * 60}")
    for sp, recs in sorted(dupe_storage_paths.items()):
        print(f"\n  Path: {sp}  ({len(recs)} records)")
        rows = []
        for r in recs:
            rows.append([r["id"][:12] + "...", r.get("title", "—")[:35], r["created_at"][:19]])
        print(tabulate(rows, headers=["Record ID", "Title", "Created At"], tablefmt="simple"))

if total_exact_dupes == 0:
    print("\nNo exact duplicates found.")

EXACT DUPLICATES

Check                          Groups Found
---------------------------  --------------
Duplicate filenames in DB                 0
Duplicate storage_path refs               0

Total exact duplicate groups: 0  (0 extra records)

No exact duplicates found.


---
## Potential Duplicates

These are **heuristic matches** — files that look similar based on metadata but may not be actual duplicates. Review manually before taking action.

| Check | What it finds | Confidence |
|-------|---------------|------------|
| Same file size + MIME type | Different filenames but identical `file_size_bytes` + `mime_type` | **Medium** — could be renamed copies or coincidence |
| Same R2 object size | Different R2 keys with identical file sizes (>1 MB) | **Low** — different videos can happen to be the same size |

In [10]:
# ══════════════════════════════════════════════
# POTENTIAL DUPLICATES (heuristic — review manually)
# ══════════════════════════════════════════════

# ── 1. Duplicate file_size + mime_type (excluding exact filename matches) ──
size_type_groups = defaultdict(list)
for r in db_records:
    size = r.get("file_size_bytes")
    mime = r.get("mime_type")
    if size and size > 0:
        size_type_groups[(size, mime)].append(r)

# Filter to groups with >1 record, and exclude groups where all filenames are the same
# (those are already caught by exact duplicates above)
dupe_size_type = {}
for k, recs in size_type_groups.items():
    if len(recs) > 1:
        filenames = {r.get("original_filename") for r in recs}
        if len(filenames) > 1:  # at least 2 different filenames
            dupe_size_type[k] = recs

# ── 2. Duplicate R2 objects by file size (videos >1 MB) ──
r2_size_groups = defaultdict(list)
for key in r2_video_keys:
    size = r2_objects[key]
    if size > 1_000_000:  # >1 MB to avoid false positives
        r2_size_groups[size].append(key)

dupe_r2_sizes = {sz: keys for sz, keys in r2_size_groups.items() if len(keys) > 1}

# ── Summary ──
print("=" * 60)
print("POTENTIAL DUPLICATES (review manually)")
print("=" * 60)
print()

potential_rows = [
    ["Same file size + MIME type", len(dupe_size_type)],
    ["Same R2 object size (>1 MB)", len(dupe_r2_sizes)],
]
print(tabulate(potential_rows, headers=["Check", "Groups Found"], tablefmt="simple"))

total_potential_dupes = len(dupe_size_type) + len(dupe_r2_sizes)
print(f"\nTotal potential duplicate groups: {total_potential_dupes}")

# ── Detailed output ──

if dupe_size_type:
    print(f"\n{'─' * 60}")
    print(f"SAME FILE SIZE + MIME TYPE ({len(dupe_size_type)} groups)")
    print(f"{'─' * 60}")
    for (size, mime), recs in sorted(dupe_size_type.items(), key=lambda x: -x[0][0]):
        print(f"\n  {format_bytes(size)} / {mime or 'unknown'}  ({len(recs)} records)")
        rows = []
        for r in sorted(recs, key=lambda x: x["created_at"]):
            uid = uuid_from_path(r["storage_path"])
            rows.append([uid[:12] + "...", r.get("title", "—")[:35], r.get("original_filename", "—")[:30], r["created_at"][:19]])
        print(tabulate(rows, headers=["UUID", "Title", "Filename", "Created At"], tablefmt="simple"))

if dupe_r2_sizes:
    print(f"\n{'─' * 60}")
    print(f"SAME R2 OBJECT SIZE ({len(dupe_r2_sizes)} groups)")
    print(f"{'─' * 60}")
    for size, keys in sorted(dupe_r2_sizes.items(), reverse=True):
        print(f"\n  File size: {format_bytes(size)}  ({len(keys)} objects)")
        rows = []
        for key in sorted(keys):
            uid = uuid_from_path(key)
            title = db_by_uuid[uid].get("title", "—")[:35] if uid in db_by_uuid else "— (no DB record)"
            fname = db_by_uuid[uid].get("original_filename", "—")[:30] if uid in db_by_uuid else "—"
            rows.append([uid[:12] + "...", title, fname])
        print(tabulate(rows, headers=["UUID", "Title", "Filename"], tablefmt="simple"))

if total_potential_dupes == 0:
    print("\nNo potential duplicates found.")

POTENTIAL DUPLICATES (review manually)

Check                          Groups Found
---------------------------  --------------
Same file size + MIME type              124
Same R2 object size (>1 MB)             125

Total potential duplicate groups: 249

────────────────────────────────────────────────────────────
SAME FILE SIZE + MIME TYPE (124 groups)
────────────────────────────────────────────────────────────

  788.9 MB / video/quicktime  (2 records)
UUID             Title                         Filename                        Created At
---------------  ----------------------------  ------------------------------  -------------------
5b828dfc-757...  2025_07_27_02_23_36_IMG_8324  2025_07_27_02_23_36_IMG_8324.M  2026-03-22T20:59:13
63b47188-bb9...  IMG_8324                      IMG_8324.MOV                    2026-03-22T22:14:50

  706.2 MB / video/mp4  (2 records)
UUID             Title                         Filename                        Created At
---------------  --------

---
## Local Files vs Database

Cross-references video files on disk (`C:\Users\minds\Videos\DANCE\*`) against `original_filename` in the database to find:

- **Not uploaded** — local files with no matching DB record (need to be uploaded)
- **In DB only** — DB records whose `original_filename` doesn't match any local file (uploaded from elsewhere, or local file was renamed/deleted)

In [11]:
# ══════════════════════════════════════════════
# LOCAL FILES vs DATABASE
# ══════════════════════════════════════════════
import glob as globmod
from pathlib import Path

LOCAL_DANCE_ROOT = r"C:\Users\minds\Videos\DANCE"
EXCLUDE_DIRS = {"dance-library"}
VIDEO_EXTS = {".mov", ".mp4", ".m4v", ".avi", ".mkv", ".webm"}
IMAGE_EXTS = {".heic", ".jpg", ".jpeg", ".png"}
MEDIA_EXTS = VIDEO_EXTS | IMAGE_EXTS

# ── Scan local folders ──
local_files = {}  # filename -> full path
for entry in Path(LOCAL_DANCE_ROOT).iterdir():
    if not entry.is_dir() or entry.name in EXCLUDE_DIRS:
        continue
    for f in entry.rglob("*"):
        if f.is_file() and f.suffix.lower() in MEDIA_EXTS:
            local_files[f.name] = str(f)

# ── Build DB filename set ──
db_filenames = {}  # filename -> record
for r in db_records:
    fn = r.get("original_filename")
    if fn:
        db_filenames[fn] = r

# ── Cross-reference ──
local_names = set(local_files.keys())
db_names = set(db_filenames.keys())

matched = local_names & db_names           # on disk AND in DB
not_uploaded = local_names - db_names      # on disk but NOT in DB
db_only = db_names - local_names           # in DB but NOT on disk

# ── Summary ──
print("=" * 60)
print("LOCAL FILES vs DATABASE")
print("=" * 60)
print()

local_rows = [
    ["Local media files scanned", len(local_files)],
    ["DB records with filename", len(db_filenames)],
    ["", ""],
    ["Matched (on disk + in DB)", len(matched)],
    ["Not uploaded (on disk only)", len(not_uploaded)],
    ["In DB only (no local file)", len(db_only)],
]
print(tabulate(local_rows, headers=["", "Count"], tablefmt="simple"))

# ── Detailed: Not uploaded ──
if not_uploaded:
    print(f"\n{'─' * 60}")
    print(f"NOT UPLOADED — on disk but no DB record ({len(not_uploaded)})")
    print(f"{'─' * 60}")

    # Group by parent folder
    by_folder = defaultdict(list)
    for fn in sorted(not_uploaded):
        folder = Path(local_files[fn]).parent.name
        by_folder[folder].append(fn)

    for folder, files in sorted(by_folder.items()):
        print(f"\n  {folder}/ ({len(files)} files)")
        rows = []
        for fn in files:
            fpath = Path(local_files[fn])
            size = format_bytes(fpath.stat().st_size)
            rows.append([fn[:45], size])
        print(tabulate(rows, headers=["Filename", "Size"], tablefmt="simple"))

# ── Detailed: In DB only ──
if db_only:
    print(f"\n{'─' * 60}")
    print(f"IN DB ONLY — no matching local file ({len(db_only)})")
    print(f"{'─' * 60}")
    rows = []
    for fn in sorted(db_only):
        r = db_filenames[fn]
        size = format_bytes(r["file_size_bytes"]) if r.get("file_size_bytes") else "—"
        rows.append([fn[:40], r.get("title", "—")[:30], size, r["created_at"][:19]])
    print(tabulate(rows, headers=["Filename", "Title", "DB Size", "Created At"], tablefmt="simple"))

if not not_uploaded and not db_only:
    print("\nLocal files and database are fully in sync!")

LOCAL FILES vs DATABASE

                               Count
---------------------------  -------
Local media files scanned          0
DB records with filename         481

Matched (on disk + in DB)          0
Not uploaded (on disk only)        0
In DB only (no local file)       481

────────────────────────────────────────────────────────────
IN DB ONLY — no matching local file (481)
────────────────────────────────────────────────────────────
Filename                                  Title                           DB Size    Created At
----------------------------------------  ------------------------------  ---------  -------------------
2024_06_02_20_44_34_IMG_0462.MOV          2024_06_02_20_44_34_IMG_0462    15.1 MB    2026-03-22T10:11:41
2024_06_02_22_49_11_IMG_0463.MOV          2024_06_02_22_49_11_IMG_0463    275.4 MB   2026-03-22T10:18:17
2024_07_01_21_53_50_IMG_0847.MP4          2024_07_01_21_53_50_IMG_0847    38.1 MB    2026-03-22T10:12:44
2024_07_04_10_14_14_IMG_0911.MP4  

In [12]:
# ── Delete duplicate DB records (keep oldest, remove newer duplicates) ──
# For filename duplicates: keeps the record with the earliest created_at.
# Deletes the duplicate DB record AND its R2 video/thumbnail files.
# Uncomment and run ONLY after reviewing the duplicate detection output above.

# for fn, recs in dupe_filenames.items():
#     sorted_recs = sorted(recs, key=lambda x: x["created_at"])
#     keep = sorted_recs[0]
#     dupes_to_remove = sorted_recs[1:]
#     print(f"\n  Filename: {fn}")
#     print(f"    Keeping:  {keep['id'][:12]}... (created {keep['created_at'][:19]})")
#     for r in dupes_to_remove:
#         uid = uuid_from_path(r["storage_path"])
#         # Delete from DB
#         sb.table("media").delete().eq("id", r["id"]).execute()
#         print(f"    Deleted DB record: {r['id'][:12]}... (created {r['created_at'][:19]})")
#         # Delete video from R2 if it exists
#         if uid in r2_videos_by_uuid:
#             r2.delete_object(Bucket=R2_BUCKET_NAME, Key=r2_videos_by_uuid[uid])
#             print(f"      Deleted R2 video: {r2_videos_by_uuid[uid]}")
#         # Delete thumbnail from R2 if it exists
#         if uid in r2_thumbs_by_uuid:
#             r2.delete_object(Bucket=R2_BUCKET_NAME, Key=r2_thumbs_by_uuid[uid])
#             print(f"      Deleted R2 thumb: {r2_thumbs_by_uuid[uid]}")
# print("Done.")

---
## Manual Cleanup (run cells below only if needed)

**These cells delete data. Review the output above carefully before running.**

In [13]:
# ── Delete orphaned R2 files (orphaned_pair + orphaned_video + orphaned_thumb) ──
# Uncomment and run ONLY after reviewing the breakdown above.

# orphan_keys = []
# for uid in states["orphaned_pair"] + states["orphaned_video"]:
#     if uid in r2_videos_by_uuid:
#         orphan_keys.append(r2_videos_by_uuid[uid])
# for uid in states["orphaned_pair"] + states["orphaned_thumb"]:
#     if uid in r2_thumbs_by_uuid:
#         orphan_keys.append(r2_thumbs_by_uuid[uid])
#
# print(f"About to delete {len(orphan_keys)} orphaned file(s) from R2...")
# for key in orphan_keys:
#     r2.delete_object(Bucket=R2_BUCKET_NAME, Key=key)
#     print(f"  Deleted: {key}")
# print("Done.")

In [14]:
# ── Delete ghost DB records (no video or thumbnail in R2) ──
# Also handles missing_video records (DB + thumb but no video = broken).
# Uncomment and run ONLY after reviewing the breakdown above.

# ghost_uuids = states["ghost_record"] + states["missing_video"]
# for uid in ghost_uuids:
#     r = db_by_uuid[uid]
#     sb.table("media").delete().eq("id", r["id"]).execute()
#     print(f"  Deleted DB record: {r['id']} ({r['title']})")
#     # Also clean up orphaned thumb if it exists
#     if uid in r2_thumbs_by_uuid:
#         r2.delete_object(Bucket=R2_BUCKET_NAME, Key=r2_thumbs_by_uuid[uid])
#         print(f"    Deleted orphaned thumb: {r2_thumbs_by_uuid[uid]}")
# print("Done.")

In [15]:
# ── Clear thumbnail_path for "missing thumbnail" records ──
# These have a video but no thumb — clearing the path makes the state explicit.
# (Use generate_thumbnails.ipynb to regenerate them afterward.)

# for uid in states["missing_thumb"]:
#     r = db_by_uuid[uid]
#     if r.get("thumbnail_path"):
#         sb.table("media").update({"thumbnail_path": None}).eq("id", r["id"]).execute()
#         print(f"  Cleared thumbnail_path: {r['id']} ({r['title']})")
# print("Done.")